# A/B: ¿MTP acelera sin cambiar el resultado? (`Qwen/Qwen3.5-0.8B`, MATH-500, Colab L4)

En la carrera de motores, vLLM + MTP (las cabezas de predicción multi-token de Qwen3.5) generó **1.52× más rápido**
(32 preguntas, 2k tokens). Antes de meterlo al pipeline hay que ver tres cosas en una corrida real:
1. **El corte temprano de loops sigue funcionando.** Con speculative decoding cada petición ocupa varias filas de
   logits; antes `loop_kill_v2` se apagaba solo en ese caso. Ya está corregido (y el pipeline se niega a arrancar MTP
   sin el corte). En el log debe salir `early_loop_stop=True spec={'method': 'mtp', ...}`.
2. **Mismo resultado:** aciertos y tasa de loops iguales a la corrida sin MTP (10.0% / 85.0%).
3. **Más rápido con lote grande** (hasta 150 a la vez), no solo con 32.

Corre los **mismos 200 problemas, mismas semillas, mismo sampling `qwen`, 16k, KV en bf16** que
`loopcheck_math500_08b_colab.ipynb`, ahora con MTP, **sin la caché de outputs** (si no, solo leería los viejos).
Compara contra la corrida sin MTP que ya está en Drive (`wsi_runs/loopcheck_math500_qwen35-08b/qwen/`). ~15–25 min.

Resultado final: `MyDrive/wsi_runs/FINALES/mtp_ab_math500_qwen35-08b.json`. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
N = 200          # mismos problemas que el loopcheck
MAX_TOK = 16384
RUN_NAME = "mtp_ab_math500_qwen35-08b"
REF_RUN = "loopcheck_math500_qwen35-08b"   # la corrida sin MTP (perfil qwen)
SPEC_TOKENS = 2
REPO = "/content/wsi"
BASE = "/content/base"
WORK = "/content/work"
S = f"{REPO}/scripts/discover_math500_blind.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm sympy antlr4-python3-runtime==4.11.1 hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
REF = f"/content/drive/MyDrive/wsi_runs/{REF_RUN}/qwen"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
VLLM_CACHE = "/content/drive/MyDrive/wsi_runs/_vllm_cache"
for d in (DRIVE, FINALES, VLLM_CACHE, WORK):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{REF}/outputs_elicit.jsonl"), f"falta la corrida sin MTP en {REF}"
W = f"{WORK}/mtp"
if os.path.isdir(DRIVE) and os.listdir(DRIVE) and not os.path.isdir(W):  # resume
    shutil.copytree(DRIVE, W)
os.environ.pop("WSI_BASE_CACHE_DIR", None)   # NO cache: it would return the old no-MTP outputs
os.environ["VLLM_CACHE_ROOT"] = VLLM_CACHE
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WSI_NO_FP8_KV"] = "1"            # same KV dtype as the reference run
os.environ["WSI_SPEC"] = "mtp"
os.environ["WSI_SPEC_TOKENS"] = str(SPEC_TOKENS)
print("referencia:", sorted(os.listdir(REF)))

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
import time, sys
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9

def run_step(cmd, env=None):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

env = {**os.environ, "WSI_SYNC_DIR": DRIVE}
run_step(["python", "-u", S, "prepare", "--work", W, "--base-dir", BASE, "--model-repo", MODEL,
          "--sampling", "qwen", "--n-elicit", str(N), "--n-dev", "0", "--n-test", "0",
          "--max-tokens-discover", str(MAX_TOK)], env)
# same problems and seeds as the reference run, or the comparison is meaningless
ref_req = [json.loads(l) for l in open(f"{REF}/requests_elicit.jsonl")]
new_req = [json.loads(l) for l in open(f"{W}/requests_elicit.jsonl")]
assert [(r["uid"], r["seed"], r["prompt_ids"]) for r in ref_req] == [(r["uid"], r["seed"], r["prompt_ids"]) for r in new_req], \
    "los problemas o las semillas no coinciden con la corrida de referencia"
print("mismos 200 problemas y semillas que la referencia: OK")
run_step(["python", "-u", S, "elicit", "--work", W, "--force"], env)

In [ ]:
sys.path.insert(0, f"{REPO}/scripts")
from pathlib import Path
import math, statistics
import discover_math500_blind as m

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return [round((c - h) / d, 4), round((c + h) / d, 4)]

ref_w = Path("/content/ref_qwen")
if not ref_w.exists():
    shutil.copytree(REF, ref_w)
cfg = json.loads((ref_w / "config.json").read_text()); cfg["base_dir"] = BASE
(ref_w / "config.json").write_text(json.dumps(cfg))
arms = {"no_mtp": ref_w, "mtp": Path(W)}
res = {"model": MODEL, "n": N, "max_tokens": MAX_TOK, "spec_tokens": SPEC_TOKENS, "arms": {}}
rows = {}
for name, w in arms.items():
    s = m.score_math(w, "requests_elicit.jsonl", "outputs_elicit.jsonl")
    rows[name] = {r["id"]: r for r in s["rows"]}
    outs = [json.loads(l) for l in open(w / "outputs_elicit.jsonl")]
    n = s["n"]
    a = {"n": n}
    for k, key in (("accuracy", "correct"), ("loop_killed", "looped_killed"), ("truncated", "truncated")):
        c = sum(r[key] for r in s["rows"]); a[k] = round(c / n, 4); a[k + "_ci95"] = wilson(c, n)
    a["finish_length"] = sum(o["finish_reason"] == "length" for o in outs)
    a["total_tokens"] = sum(len(o["token_ids"]) for o in outs)
    looped = [r["n_tokens"] for r in s["rows"] if r["looped_killed"]]
    a["looped_median_tokens"] = int(statistics.median(looped)) if looped else None  # far below 16k = early stop works
    t = json.loads((w / "timing_fast.json").read_text()) if (w / "timing_fast.json").exists() else {}
    a["elicit_s"] = t.get("elicit")
    res["arms"][name] = a
    print(f"{name:7s} acc {a['accuracy']:.1%} {a['accuracy_ci95']} | loop {a['loop_killed']:.1%} | trunc {a['truncated']:.1%} "
          f"| hit 16k {a['finish_length']} | loops stopped at median {a['looped_median_tokens']} tok "
          f"| {a['total_tokens']:,} tok in {a['elicit_s']} s")
ids = sorted(set(rows["no_mtp"]) & set(rows["mtp"]))
A, B = rows["no_mtp"], rows["mtp"]
res["paired"] = {"n": len(ids),
                 "mtp_right_only": sum(B[i]["correct"] and not A[i]["correct"] for i in ids),
                 "no_mtp_right_only": sum(A[i]["correct"] and not B[i]["correct"] for i in ids),
                 "loop_both": sum(A[i]["looped_killed"] and B[i]["looped_killed"] for i in ids),
                 "loop_mtp_only": sum(B[i]["looped_killed"] and not A[i]["looped_killed"] for i in ids),
                 "loop_no_mtp_only": sum(A[i]["looped_killed"] and not B[i]["looped_killed"] for i in ids)}
ra, rb = res["arms"]["no_mtp"], res["arms"]["mtp"]
if ra["elicit_s"] and rb["elicit_s"]:
    res["speedup_wall"] = round(ra["elicit_s"] / rb["elicit_s"], 3)
    res["speedup_tok_per_s"] = round((rb["total_tokens"] / rb["elicit_s"]) / (ra["total_tokens"] / ra["elicit_s"]), 3)
kill_ok = rb["looped_median_tokens"] is not None and rb["looped_median_tokens"] < MAX_TOK * 0.75
same = abs(rb["loop_killed"] - ra["loop_killed"]) <= 0.06 and abs(rb["accuracy"] - ra["accuracy"]) <= 0.05
res["early_stop_works_with_mtp"] = kill_ok
res["verdict"] = ("adopt MTP" if kill_ok and same and res.get("speedup_wall", 0) > 1.15 else
                  "do not adopt yet: " + ", ".join(x for x, bad in (("early stop off", not kill_ok),
                                                                     ("results differ", not same),
                                                                     ("no real speedup", res.get("speedup_wall", 0) <= 1.15)) if bad))
print("pareado:", res["paired"])
print("speedup (pared):", res.get("speedup_wall"), "| speedup (tok/s):", res.get("speedup_tok_per_s"))
print("veredicto:", res["verdict"])

In [ ]:
res["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "mtp_ab_math500_08b_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
